# 05 — Visualize Results

Interactive 3D visualization, uncertainty maps, and export for publication.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from research_ct.io.volume_saver import Load_From_Numpy
from research_ct.analysis.material_stats import Compute_Material_Statistics, Print_Material_Report
from research_ct.analysis.uncertainty_maps import Compute_Uncertainty, Compute_Margin

# Load all outputs
Processed = Load_From_Numpy("../data/processed/preprocessed_volume.npz")
Labels = Load_From_Numpy("../data/output/gmm_labels.npz")
Probs = Load_From_Numpy("../data/output/gmm_probabilities.npz")

print(f"Processed: {Processed.shape}")
print(f"Labels: {Labels.shape}")
print(f"Probabilities: {Probs.shape}")

## Material Statistics Report

Print quantitative summary of segmented materials.

In [ ]:
Num_Classes = int(Labels.max() + 1)
Stats = Compute_Material_Statistics(Processed, Labels, Num_Classes)
Print_Material_Report(Stats)

## Uncertainty Maps

Identify regions where the segmentation is ambiguous. High entropy = high uncertainty.

In [ ]:
Entropy, Max_Prob = Compute_Uncertainty(Probs)
Margin = Compute_Margin(Probs, Top_K=2)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Entropy
im0 = axes[0].imshow(Entropy[Processed.shape[0]//2], cmap='hot')
axes[0].set_title('Entropy (Uncertainty)')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

# Max probability
im1 = axes[1].imshow(Max_Prob[Processed.shape[0]//2], cmap='viridis')
axes[1].set_title('Max Probability (Confidence)')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

# Margin
im2 = axes[2].imshow(Margin[Processed.shape[0]//2], cmap='coolwarm')
axes[2].set_title('Margin (Top 2 Classes)')
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.suptitle('Uncertainty Maps — Middle Slice', fontsize=14)
plt.tight_layout()
plt.show()

## 3D Visualization with Napari

Launch interactive viewer. Toggle layers, adjust opacity, inspect in 3D.

In [ ]:
from research_ct.visualization.napari_viewer import launch_napari_viewer

# This will open a napari window
launch_napari_viewer(Processed, Labels, Probs)

## Export Probability Videos

Create videos showing class probability through Z slices. Useful for presentations.

In [ ]:
from research_ct.visualization.export import export_probability_video

# Export each class probability as video
for k in range(Num_Classes):
    Output_Path = f"../data/output/probability_class_{k}.mp4"
    export_probability_video(Probs, Output_Path, Class_Index=k, Fps=15)
    print(f"Exported: {Output_Path}")

## Export Colored Label Stacks

Save segmentation as colored TIFF stack for external tools (ImageJ, etc.).

In [ ]:
from research_ct.visualization.export import export_label_colors

# Define colors for each material
Color_Map = {
    0: (0, 0, 0),         # Air/background
    1: (255, 228, 181),   # Paper
    2: (47, 79, 79),      # Ink
    3: (139, 69, 19),     # Cover
}

export_label_colors(Labels, "../data/output/segmentation_colored.tif", Color_Map)
print("Exported colored segmentation")

## Component Distribution Plots

Generate publication-quality GMM decomposition figure.

In [ ]:
from research_ct.visualization.plot_distributions import plot_gmm_components
from research_ct.segmentation.gmm_fitter import Gmm_Fitter

# Re-fit on sample for plotting
Sample = Processed.ravel()[np.random.choice(Processed.size, 200_000, replace=False)]
Fitter = Gmm_Fitter(Min_Components=Num_Classes, Max_Components=Num_Classes)
Fitter.Fit(Sample.reshape(-1, 1), Verbose=False)

fig = plot_gmm_components(
    Sample,
    Fitter.Model,
    Output_Path="../data/output/gmm_decomposition.png"
)
plt.show()
print("Saved: ../data/output/gmm_decomposition.png")

## Summary

At this point you should have:
- [ ] Segmented labels for the full volume
- [ ] Probability maps for uncertainty quantification
- [ ] Material statistics (voxel counts, fractions, intensities)
- [ ] Diagnostic plots (edge strength, histograms)
- [ ] 3D visualization in napari
- [ ] Export files for sharing/presentation

Next steps:
- Tune HMRF beta if results are too noisy or over-smoothed
- Try hierarchical splitting if ink/paper separation is poor
- Proceed to geometric page extraction (Charles' approach)